# MIMIC-IV: Length of Stay > 7 Days — Binary Classification Pipeline

This notebook builds a complete ML pipeline for predicting whether a hospital stay
exceeds 7 days using the MIMIC-IV Clinical Database (v3.1).

**Pipeline overview:**
1. Data Exploration
2. Cohort Selection & Target Definition
3. Raw Feature Extraction (vitals, labs, comorbidities)
4. Train / Val / Test Split (before any fitting!)
5. Preprocessing (fit on train only)
6. Exploratory Data Analysis & Plots
7. MLP Model (PyTorch)
8. Evaluation & Fairness Check

**Data requirement:** MIMIC-IV v3.1 files under `data/raw/mimiciv/3.1/`.

In [ ]:
import os
import time
import glob
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)

PIPELINE_START = time.time()

# --- Path configuration ---
# Adjust BASE_DIR if your notebook is not in notebooks/
PROJ_DIR = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
DATA_RAW = PROJ_DIR / 'data' / 'raw'
FIGURES  = PROJ_DIR / 'figures'
SPLITS   = PROJ_DIR / 'data' / 'splits'
PROCESSED = PROJ_DIR / 'data' / 'processed'

for d in [FIGURES, SPLITS, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

# Auto-detect MIMIC root (handles nested physionet download structure)
# Also check the main repo path in case we're running from a worktree
MAIN_REPO_RAW = Path('/Users/maxim/Documents/Code/TeamProject/teamproject_heinzl_FSS26/data/raw')
candidates = [
    DATA_RAW / 'mimiciv' / '3.1',
    MAIN_REPO_RAW / 'mimiciv' / '3.1',
    DATA_RAW / 'physionet.org' / 'files' / 'mimiciv' / '3.1',
    DATA_RAW / 'hosp',
    MAIN_REPO_RAW / 'hosp',
    DATA_RAW,
    MAIN_REPO_RAW,
]
MIMIC_ROOT = None
for c in candidates:
    if (c / 'hosp').is_dir() or (c.name == 'hosp' and c.is_dir()):
        MIMIC_ROOT = c if c.name != 'hosp' else c.parent
        break

if MIMIC_ROOT is None:
    raise FileNotFoundError(
        f"Could not find MIMIC-IV data under {DATA_RAW}. "
        "Expected hosp/ and icu/ subdirectories."
    )

HOSP = MIMIC_ROOT / 'hosp'
ICU  = MIMIC_ROOT / 'icu'
print(f"MIMIC root : {MIMIC_ROOT}")
print(f"HOSP tables: {HOSP}")

---
## Step 1 — Data Exploration

We detect file formats automatically (`.csv`, `.csv.gz`, `.parquet`) and print
basic statistics for each table. For very large files (`chartevents`, `labevents`)
we only load a sample for initial exploration.

In [ ]:
def find_table(directory: Path, name: str) -> Path:
    """Find a table file, checking .csv.gz, .csv, and .parquet extensions."""
    for ext in ['.csv.gz', '.csv', '.parquet']:
        p = directory / f"{name}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Table '{name}' not found in {directory}")


def load_table(path: Path, nrows=None, usecols=None, dtype=None):
    """Load a table from .csv, .csv.gz, or .parquet."""
    if path.suffix == '.parquet':
        df = pd.read_parquet(path, columns=usecols)
        if nrows is not None:
            df = df.head(nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, usecols=usecols, dtype=dtype,
                         low_memory=False)
    return df


def explore_table(directory: Path, name: str, nrows=None):
    """Print exploration stats for a table."""
    path = find_table(directory, name)
    label = f"{name} ({path.suffix})"
    if nrows:
        label += f" [first {nrows:,} rows]"
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    df = load_table(path, nrows=nrows)
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    print(f"\nMissing %:\n{missing_pct[missing_pct > 0]}")
    print(f"\nHead:")
    display(df.head(3))
    return df

In [ ]:
# Explore small-to-medium tables fully
df_adm_explore = explore_table(HOSP, 'admissions')
df_pat_explore = explore_table(HOSP, 'patients')
df_diag_explore = explore_table(HOSP, 'diagnoses_icd')
df_icu_explore = explore_table(ICU, 'icustays')

In [ ]:
# Explore large tables with limited rows
print("\n" + "*"*60)
print("  LARGE FILES — sampled exploration (100k rows)")
print("*"*60)

_ = explore_table(HOSP, 'labevents', nrows=100_000)
_ = explore_table(ICU, 'chartevents', nrows=100_000)

---
## Step 2 — Cohort Selection & Target Definition

We merge admissions with patients, compute length of stay (LOS), define the
binary target (LOS > 7 days), and apply exclusion criteria.

In [ ]:
# Load full admissions and patients
df_admissions = load_table(find_table(HOSP, 'admissions'))
df_patients   = load_table(find_table(HOSP, 'patients'))

# Parse datetime columns
for col in ['admittime', 'dischtime', 'deathtime', 'edregtime', 'edouttime']:
    if col in df_admissions.columns:
        df_admissions[col] = pd.to_datetime(df_admissions[col], errors='coerce')

# Merge on subject_id
df = df_admissions.merge(df_patients, on='subject_id', how='inner')
print(f"Merged admissions × patients: {df.shape}")

# Compute LOS in days
df['los_days'] = (df['dischtime'] - df['admittime']).dt.total_seconds() / 86400

# Binary target
df['los_target'] = (df['los_days'] > 7).astype(int)

print(f"\nLOS stats (days):")
print(df['los_days'].describe().round(2))

In [ ]:
# --- Exclusion criteria (applied in order, with counts) ---

n_start = len(df)
print(f"Starting cohort: {n_start:,}")

# 1. Remove LOS < 4 hours
mask = df['los_days'] >= (4 / 24)
print(f"Excluded LOS < 4h: {(~mask).sum():,}")
df = df[mask].copy()

# 2. Remove age < 18
mask = df['anchor_age'] >= 18
print(f"Excluded age < 18: {(~mask).sum():,}")
df = df[mask].copy()

# 3. Keep only first admission per patient
n_before_first = len(df)
df = df.sort_values('admittime').groupby('subject_id').first().reset_index()
n_removed_multi = n_before_first - len(df)
print(f"Excluded repeat admissions (keep first only): {n_removed_multi:,}")

> **Note on first-visit-only filter:** This ensures statistical independence between
> samples — each patient contributes exactly one observation. If multiple admissions
> are needed later (e.g., for longitudinal models), `GroupShuffleSplit` on `subject_id`
> would handle data leakage, but first-visit-only is the cleaner default for a
> tabular baseline.

In [ ]:
# Drop leakage columns
leakage_cols = [
    'dischtime', 'admittime', 'deathtime', 'discharge_location',
    'edregtime', 'edouttime', 'hospital_expire_flag',
]
# Keep admittime temporarily — we need it for 24h feature windows
# We'll drop it after feature extraction
drop_now = [c for c in leakage_cols if c in df.columns and c != 'admittime']
# Save admittime for later use
admit_times = df[['hadm_id', 'admittime']].copy() if 'admittime' in df.columns else None
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

print(f"Dropped leakage columns: {drop_now}")
print(f"\nFinal cohort size: {len(df):,}")
print(f"\nClass distribution:")
counts = df['los_target'].value_counts().sort_index()
pcts = (counts / len(df) * 100).round(1)
for label, count, pct in zip(['LOS ≤ 7d', 'LOS > 7d'], counts.values, pcts.values):
    print(f"  {label}: {count:>8,}  ({pct}%)")

---
## Step 3 — Raw Feature Extraction

We extract and aggregate features from three sources:
- **Vitals** from `chartevents` (first 24h after admission)
- **Labs** from `labevents` (first 24h)
- **Comorbidities** from `diagnoses_icd` (Charlson Index + ICD chapter flags)

This is pure transformation — no fitting, no statistics derived from the full dataset.

### 3a — Vital Signs (chartevents, first 24h)

We load chartevents in chunks, filtering to the needed `itemid`s and
the first 24 hours after each admission. This keeps memory manageable
even for the ~5+ GB file.

In [ ]:
# Vital sign itemid mapping
VITAL_ITEMS = {
    220045: 'heart_rate',
    220277: 'spo2',
    223762: 'temperature',
    220210: 'resp_rate',
    220739: 'gcs_total',
    220179: 'sbp',
    220180: 'dbp',
    220052: 'map',
}
VITAL_IDS = set(VITAL_ITEMS.keys())

# Build admit-time lookup for 24h window filtering
admit_lookup = admit_times.set_index('hadm_id')['admittime'].to_dict()
valid_hadm_ids = set(df['hadm_id'].values)

print("Loading chartevents in chunks (filtering to first 24h vitals)...")
chart_path = find_table(ICU, 'chartevents')

chunks = []
use_cols = ['hadm_id', 'itemid', 'charttime', 'valuenum']

# Determine available columns first
sample = load_table(chart_path, nrows=5)
available_cols = [c for c in use_cols if c in sample.columns]
# Some versions use 'value' instead of 'valuenum'
if 'valuenum' not in sample.columns and 'value' in sample.columns:
    available_cols = [c if c != 'valuenum' else 'value' for c in available_cols]

reader = pd.read_csv(
    chart_path,
    usecols=available_cols,
    chunksize=500_000,
    dtype={'hadm_id': 'Int64', 'itemid': 'Int64'},
    low_memory=False,
)

n_chunks = 0
for chunk in reader:
    n_chunks += 1
    # Rename 'value' → 'valuenum' if needed
    if 'value' in chunk.columns and 'valuenum' not in chunk.columns:
        chunk = chunk.rename(columns={'value': 'valuenum'})
        chunk['valuenum'] = pd.to_numeric(chunk['valuenum'], errors='coerce')

    # Filter to our cohort and vital itemids
    chunk = chunk[chunk['hadm_id'].isin(valid_hadm_ids) & chunk['itemid'].isin(VITAL_IDS)].copy()
    if len(chunk) == 0:
        continue

    # Filter to first 24h
    chunk['charttime'] = pd.to_datetime(chunk['charttime'], errors='coerce')
    chunk['admittime'] = chunk['hadm_id'].map(admit_lookup)
    chunk = chunk[
        (chunk['charttime'] >= chunk['admittime']) &
        (chunk['charttime'] <= chunk['admittime'] + pd.Timedelta(hours=24))
    ]
    chunk = chunk.dropna(subset=['valuenum'])
    if len(chunk) > 0:
        chunks.append(chunk[['hadm_id', 'itemid', 'valuenum']])

    if n_chunks % 20 == 0:
        print(f"  Processed {n_chunks} chunks...")

print(f"  Done — processed {n_chunks} chunks total.")

df_vitals_raw = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=['hadm_id','itemid','valuenum'])
df_vitals_raw['vital_name'] = df_vitals_raw['itemid'].map(VITAL_ITEMS)
print(f"  Vital records after filtering: {len(df_vitals_raw):,}")

In [ ]:
# Aggregate vitals: mean, min, max, std per hadm_id × vital
vitals_agg = (
    df_vitals_raw
    .groupby(['hadm_id', 'vital_name'])['valuenum']
    .agg(['mean', 'min', 'max', 'std'])
    .reset_index()
)

# Pivot to wide format
vitals_wide = vitals_agg.pivot_table(
    index='hadm_id',
    columns='vital_name',
    values=['mean', 'min', 'max', 'std'],
)
vitals_wide.columns = [f"{vital}_{stat}" for stat, vital in vitals_wide.columns]
vitals_wide = vitals_wide.reset_index()

# Add binary _missing flags for each vital
all_hadm = df[['hadm_id']].copy()
vitals_wide = all_hadm.merge(vitals_wide, on='hadm_id', how='left')

for vital in VITAL_ITEMS.values():
    mean_col = f"{vital}_mean"
    missing_col = f"{vital}_missing"
    if mean_col in vitals_wide.columns:
        vitals_wide[missing_col] = vitals_wide[mean_col].isna().astype(int)
    else:
        vitals_wide[missing_col] = 1

print(f"Vitals feature matrix: {vitals_wide.shape}")
print(f"Missing rates (% of patients with no measurement):")
for vital in VITAL_ITEMS.values():
    mc = f"{vital}_missing"
    if mc in vitals_wide.columns:
        pct = vitals_wide[mc].mean() * 100
        print(f"  {vital}: {pct:.1f}%")

### 3b — Lab Values (labevents, first 24h)

In [ ]:
# Lab itemid mapping
LAB_ITEMS = {
    50912: 'creatinine',
    51222: 'hemoglobin',
    51301: 'wbc',
    50983: 'sodium',
    50971: 'potassium',
    50813: 'lactate',
    50882: 'bicarbonate',
    51006: 'bun',
}
LAB_IDS = set(LAB_ITEMS.keys())

print("Loading labevents in chunks (filtering to first 24h labs)...")
lab_path = find_table(HOSP, 'labevents')

# Determine columns
lab_sample = load_table(lab_path, nrows=5)
lab_use_cols = ['hadm_id', 'itemid', 'charttime', 'valuenum']
lab_available = [c for c in lab_use_cols if c in lab_sample.columns]
# Fallback: storetime if charttime not present
time_col = 'charttime' if 'charttime' in lab_sample.columns else 'storetime'
if time_col != 'charttime':
    lab_available = [c if c != 'charttime' else time_col for c in lab_available]

lab_chunks = []
reader = pd.read_csv(
    lab_path,
    usecols=lab_available,
    chunksize=500_000,
    dtype={'hadm_id': 'Int64', 'itemid': 'Int64'},
    low_memory=False,
)

n_chunks = 0
for chunk in reader:
    n_chunks += 1
    if time_col != 'charttime' and time_col in chunk.columns:
        chunk = chunk.rename(columns={time_col: 'charttime'})

    chunk = chunk.dropna(subset=['hadm_id'])
    chunk['hadm_id'] = chunk['hadm_id'].astype(int)
    chunk = chunk[chunk['hadm_id'].isin(valid_hadm_ids) & chunk['itemid'].isin(LAB_IDS)].copy()
    if len(chunk) == 0:
        continue

    chunk['charttime'] = pd.to_datetime(chunk['charttime'], errors='coerce')
    chunk['admittime'] = chunk['hadm_id'].map(admit_lookup)
    chunk = chunk[
        (chunk['charttime'] >= chunk['admittime']) &
        (chunk['charttime'] <= chunk['admittime'] + pd.Timedelta(hours=24))
    ]
    chunk = chunk.dropna(subset=['valuenum'])
    if len(chunk) > 0:
        lab_chunks.append(chunk[['hadm_id', 'itemid', 'valuenum']])

    if n_chunks % 20 == 0:
        print(f"  Processed {n_chunks} chunks...")

print(f"  Done — processed {n_chunks} chunks total.")

df_labs_raw = pd.concat(lab_chunks, ignore_index=True) if lab_chunks else pd.DataFrame(columns=['hadm_id','itemid','valuenum'])
df_labs_raw['lab_name'] = df_labs_raw['itemid'].map(LAB_ITEMS)
print(f"  Lab records after filtering: {len(df_labs_raw):,}")

In [ ]:
# Aggregate labs: mean, min, max per hadm_id × lab
labs_agg = (
    df_labs_raw
    .groupby(['hadm_id', 'lab_name'])['valuenum']
    .agg(['mean', 'min', 'max'])
    .reset_index()
)

labs_wide = labs_agg.pivot_table(
    index='hadm_id',
    columns='lab_name',
    values=['mean', 'min', 'max'],
)
labs_wide.columns = [f"{lab}_{stat}" for stat, lab in labs_wide.columns]
labs_wide = labs_wide.reset_index()

# Merge with all hadm_ids and add missing flags
labs_wide = all_hadm.merge(labs_wide, on='hadm_id', how='left')

for lab in LAB_ITEMS.values():
    mean_col = f"{lab}_mean"
    missing_col = f"{lab}_missing"
    if mean_col in labs_wide.columns:
        labs_wide[missing_col] = labs_wide[mean_col].isna().astype(int)
    else:
        labs_wide[missing_col] = 1

print(f"Labs feature matrix: {labs_wide.shape}")
for lab in LAB_ITEMS.values():
    mc = f"{lab}_missing"
    if mc in labs_wide.columns:
        print(f"  {lab} missing: {labs_wide[mc].mean()*100:.1f}%")

### 3c — Comorbidities (diagnoses_icd → Charlson Index)

In [ ]:
# Load diagnoses
df_diag = load_table(find_table(HOSP, 'diagnoses_icd'))
df_diag = df_diag[df_diag['hadm_id'].isin(valid_hadm_ids)].copy()

# Charlson comorbidity ICD-9/10 code mappings
# Each key is a Charlson component; values are (icd_version, prefix_list) tuples
CHARLSON_MAP = {
    'myocardial_infarction': {
        9: ['410', '412'],
        10: ['I21', 'I22', 'I252'],
    },
    'congestive_heart_failure': {
        9: ['39891', '4254', '4255', '4256', '4257', '4258', '4259', '428'],
        10: ['I099', 'I110', 'I130', 'I132', 'I255', 'I420', 'I425', 'I426',
             'I427', 'I428', 'I429', 'I43', 'I50', 'P290'],
    },
    'peripheral_vascular_disease': {
        9: ['0930', '4373', '440', '441', '4431', '4432', '4438', '4439',
            '4471', '5571', '5579', 'V434'],
        10: ['I70', 'I71', 'I731', 'I738', 'I739', 'I771', 'I790', 'I792',
             'K551', 'K558', 'K559', 'Z958', 'Z959'],
    },
    'cerebrovascular_disease': {
        9: ['36234', '430', '431', '432', '433', '434', '435', '436', '437', '438'],
        10: ['G45', 'G46', 'H340', 'I60', 'I61', 'I62', 'I63', 'I64',
             'I65', 'I66', 'I67', 'I68', 'I69'],
    },
    'dementia': {
        9: ['290', '2941', '3312'],
        10: ['F00', 'F01', 'F02', 'F03', 'F051', 'G30', 'G311'],
    },
    'chronic_pulmonary_disease': {
        9: ['4168', '4169', '490', '491', '492', '493', '494', '495',
            '496', '497', '498', '499', '500', '501', '502', '503',
            '504', '505', '5064', '5081', '5088'],
        10: ['I278', 'I279', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45',
             'J46', 'J47', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65',
             'J66', 'J67', 'J684', 'J701', 'J703'],
    },
    'rheumatic_disease': {
        9: ['4465', '7100', '7101', '7102', '7103', '7104', '7140', '7141', '7142', '7148', '725'],
        10: ['M05', 'M06', 'M315', 'M32', 'M33', 'M34', 'M351', 'M353', 'M360'],
    },
    'peptic_ulcer_disease': {
        9: ['531', '532', '533', '534'],
        10: ['K25', 'K26', 'K27', 'K28'],
    },
    'mild_liver_disease': {
        9: ['07022', '07023', '07032', '07033', '07044', '07054',
            '0706', '0709', '570', '571', '5733', '5734', '5738', '5739', 'V427'],
        10: ['B18', 'K700', 'K701', 'K702', 'K703', 'K709', 'K713',
             'K714', 'K715', 'K717', 'K73', 'K74', 'K760', 'K762',
             'K763', 'K764', 'K768', 'K769', 'Z944'],
    },
    'diabetes_without_complications': {
        9: ['2500', '2501', '2502', '2503', '2508', '2509'],
        10: ['E100', 'E101', 'E106', 'E108', 'E109', 'E110', 'E111',
             'E116', 'E118', 'E119', 'E120', 'E121', 'E126', 'E128',
             'E129', 'E130', 'E131', 'E136', 'E138', 'E139', 'E140',
             'E141', 'E146', 'E148', 'E149'],
    },
    'diabetes_with_complications': {
        9: ['2504', '2505', '2506', '2507'],
        10: ['E102', 'E103', 'E104', 'E105', 'E107', 'E112', 'E113',
             'E114', 'E115', 'E117', 'E122', 'E123', 'E124', 'E125',
             'E127', 'E132', 'E133', 'E134', 'E135', 'E137', 'E142',
             'E143', 'E144', 'E145', 'E147'],
    },
    'hemiplegia_paraplegia': {
        9: ['3341', '342', '343', '3440', '3441', '3442', '3443', '3444', '3445', '3446', '3449'],
        10: ['G041', 'G114', 'G801', 'G802', 'G81', 'G82', 'G830',
             'G831', 'G832', 'G833', 'G834', 'G839'],
    },
    'renal_disease': {
        9: ['40301', '40311', '40391', '40402', '40403', '40412', '40413',
            '40492', '40493', '582', '5830', '5831', '5832', '5834',
            '5836', '5837', '585', '586', '5880', 'V420', 'V451', 'V56'],
        10: ['I120', 'I131', 'N032', 'N033', 'N034', 'N035', 'N036',
             'N037', 'N052', 'N053', 'N054', 'N055', 'N056', 'N057',
             'N18', 'N19', 'N250', 'Z490', 'Z491', 'Z492', 'Z940', 'Z992'],
    },
    'malignancy': {
        9: ['140', '141', '142', '143', '144', '145', '146', '147', '148',
            '149', '150', '151', '152', '153', '154', '155', '156', '157',
            '158', '159', '160', '161', '162', '163', '164', '165', '170',
            '171', '172', '174', '175', '176', '179', '180', '181', '182',
            '183', '184', '185', '186', '187', '188', '189', '190', '191',
            '192', '193', '194', '195', '200', '201', '202', '203', '204',
            '205', '206', '207', '208', '2386'],
        10: ['C00', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08',
             'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17',
             'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26',
             'C30', 'C31', 'C32', 'C33', 'C34', 'C37', 'C38', 'C39', 'C40',
             'C41', 'C43', 'C45', 'C46', 'C47', 'C48', 'C49', 'C50', 'C51',
             'C52', 'C53', 'C54', 'C55', 'C56', 'C57', 'C58', 'C60', 'C61',
             'C62', 'C63', 'C64', 'C65', 'C66', 'C67', 'C68', 'C69', 'C70',
             'C71', 'C72', 'C73', 'C74', 'C75', 'C76', 'C81', 'C82', 'C83',
             'C84', 'C85', 'C88', 'C90', 'C91', 'C92', 'C93', 'C94', 'C95',
             'C96', 'C97'],
    },
    'severe_liver_disease': {
        9: ['4560', '4561', '4562', '5722', '5723', '5724', '5728'],
        10: ['I850', 'I859', 'I864', 'I982', 'K704', 'K711', 'K721', 'K729', 'K765', 'K766', 'K767'],
    },
    'metastatic_solid_tumor': {
        9: ['196', '197', '198', '199'],
        10: ['C77', 'C78', 'C79', 'C80'],
    },
    'aids_hiv': {
        9: ['042', '043', '044'],
        10: ['B20', 'B21', 'B22', 'B24'],
    },
}

# Charlson weights
CHARLSON_WEIGHTS = {
    'myocardial_infarction': 1, 'congestive_heart_failure': 1,
    'peripheral_vascular_disease': 1, 'cerebrovascular_disease': 1,
    'dementia': 1, 'chronic_pulmonary_disease': 1, 'rheumatic_disease': 1,
    'peptic_ulcer_disease': 1, 'mild_liver_disease': 1,
    'diabetes_without_complications': 1, 'diabetes_with_complications': 2,
    'hemiplegia_paraplegia': 2, 'renal_disease': 2, 'malignancy': 2,
    'severe_liver_disease': 3, 'metastatic_solid_tumor': 6, 'aids_hiv': 6,
}


def match_icd(code, version, prefixes):
    """Check if an ICD code matches any prefix for the given version."""
    code = str(code).strip().upper()
    for prefix in prefixes:
        if code.startswith(prefix.upper()):
            return True
    return False


# Build comorbidity flags per hadm_id
comorbidity_records = {hadm: {k: 0 for k in CHARLSON_MAP} for hadm in valid_hadm_ids}

for _, row in df_diag.iterrows():
    hadm = row['hadm_id']
    if hadm not in comorbidity_records:
        continue
    icd_code = str(row['icd_code']).strip()
    icd_ver = int(row['icd_version']) if pd.notna(row.get('icd_version')) else 9
    for comorbidity, ver_map in CHARLSON_MAP.items():
        prefixes = ver_map.get(icd_ver, [])
        if match_icd(icd_code, icd_ver, prefixes):
            comorbidity_records[hadm][comorbidity] = 1

df_charlson = pd.DataFrame.from_dict(comorbidity_records, orient='index')
df_charlson.index.name = 'hadm_id'
df_charlson = df_charlson.reset_index()

# Compute weighted Charlson score
df_charlson['charlson_score'] = sum(
    df_charlson[c] * w for c, w in CHARLSON_WEIGHTS.items()
)

print(f"Charlson comorbidity features: {df_charlson.shape}")
print(f"Mean Charlson score: {df_charlson['charlson_score'].mean():.2f}")
print(f"Comorbidity prevalence:")
for c in CHARLSON_MAP:
    print(f"  {c}: {df_charlson[c].mean()*100:.1f}%")

In [ ]:
# Top 10 ICD chapter groups as binary flags
# ICD-10 chapters by first character(s)
ICD_CHAPTERS = {
    'infectious': (['A', 'B'], ['001', '139']),
    'neoplasms': (['C', 'D0', 'D1', 'D2', 'D3', 'D4'], ['140', '239']),
    'blood_immune': (['D5', 'D6', 'D7', 'D8', 'D9'], ['280', '289']),
    'endocrine': (['E'], ['240', '279']),
    'mental': (['F'], ['290', '319']),
    'nervous': (['G'], ['320', '389']),
    'circulatory': (['I'], ['390', '459']),
    'respiratory': (['J'], ['460', '519']),
    'digestive': (['K'], ['520', '579']),
    'genitourinary': (['N'], ['580', '629']),
    'musculoskeletal': (['M'], ['710', '739']),
    'injury_poisoning': (['S', 'T'], ['800', '999']),
}


def classify_icd_chapter(code, version):
    """Return the ICD chapter name for a code."""
    code = str(code).strip().upper()
    if version == 10:
        for chapter, (prefixes_10, _) in ICD_CHAPTERS.items():
            for p in prefixes_10:
                if code.startswith(p):
                    return chapter
    else:  # ICD-9
        try:
            num = int(code[:3])
        except ValueError:
            return None
        for chapter, (_, (lo, hi)) in ICD_CHAPTERS.items():
            if int(lo) <= num <= int(hi):
                return chapter
    return None


df_diag['icd_chapter'] = df_diag.apply(
    lambda r: classify_icd_chapter(
        r['icd_code'],
        int(r['icd_version']) if pd.notna(r.get('icd_version')) else 9,
    ),
    axis=1,
)

# Find top 10 chapters
chapter_counts = df_diag.dropna(subset=['icd_chapter'])['icd_chapter'].value_counts()
top10_chapters = chapter_counts.head(10).index.tolist()
print(f"Top 10 ICD chapters: {top10_chapters}")

# Create binary flags per hadm_id
chapter_flags = (
    df_diag[df_diag['icd_chapter'].isin(top10_chapters)]
    .groupby(['hadm_id', 'icd_chapter'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)  # binary
    .reset_index()
)
chapter_flags.columns = ['hadm_id'] + [f"icd_{c}" for c in chapter_flags.columns[1:]]

# Merge with all hadm_ids
chapter_flags = all_hadm.merge(chapter_flags, on='hadm_id', how='left').fillna(0)
for c in chapter_flags.columns[1:]:
    chapter_flags[c] = chapter_flags[c].astype(int)

print(f"ICD chapter flag features: {chapter_flags.shape}")

### 3d — Merge All Features

In [ ]:
# Start with cohort base
df_features = df.copy()

# Merge vitals, labs, charlson, ICD chapter flags
df_features = df_features.merge(vitals_wide, on='hadm_id', how='left')
df_features = df_features.merge(labs_wide, on='hadm_id', how='left')
df_features = df_features.merge(df_charlson, on='hadm_id', how='left')
df_features = df_features.merge(chapter_flags, on='hadm_id', how='left')

# Encode gender → 0/1 (fixed mapping, not a fitted transformer)
if 'gender' in df_features.columns:
    df_features['gender'] = (df_features['gender'].str.upper() == 'M').astype(int)

# Drop columns not needed for modeling
drop_cols = ['los_days', 'anchor_year', 'anchor_year_group', 'dod']
drop_cols = [c for c in drop_cols if c in df_features.columns]
df_features = df_features.drop(columns=drop_cols)

print(f"Final merged feature matrix: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")

---
## ⚠️ Step 4 — Train / Val / Test Split (BEFORE any fitting)

### ⚠️ DATA LEAKAGE WARNING

**Why we split here, before imputation and scaling:**

- If we computed medians/means for imputation or scaling parameters on the full dataset
  (including test data), the test set statistics would "leak" into our preprocessing.
- This means the model would indirectly learn information about the test distribution,
  leading to **overly optimistic** performance estimates.
- **Correct approach:** Split first → fit imputers/scalers on training data only →
  transform all splits using those fitted parameters.

**Why we split on `subject_id`, not `hadm_id`:**
- Our first-visit filter already ensures a 1:1 mapping between `subject_id` and `hadm_id`.
- However, splitting on `subject_id` is the conceptually correct choice: it guarantees
  that no patient appears in both train and test, preventing patient-level leakage.
- This is especially important if the cohort definition changes in the future.

In [ ]:
# Train / Test split (80/20)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_features, groups=df_features['subject_id']))
df_train_full = df_features.iloc[train_idx].copy()
df_test       = df_features.iloc[test_idx].copy()

# Train / Val split (from train only; ~10% of full → ~12.5% of train)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=43)
train_idx2, val_idx = next(gss_val.split(df_train_full, groups=df_train_full['subject_id']))
df_train = df_train_full.iloc[train_idx2].copy()
df_val   = df_train_full.iloc[val_idx].copy()

# Save split IDs (subject_id only — safe to commit, no clinical data)
SPLITS.mkdir(parents=True, exist_ok=True)
df_train[['subject_id']].to_csv(SPLITS / 'train_ids.csv', index=False)
df_val[['subject_id']].to_csv(SPLITS / 'val_ids.csv', index=False)
df_test[['subject_id']].to_csv(SPLITS / 'test_ids.csv', index=False)

print(f"Split sizes:")
for name, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = d['los_target'].sum()
    neg = len(d) - pos
    print(f"  {name:5s}: {len(d):>7,}  (pos={pos:,} [{pos/len(d)*100:.1f}%], neg={neg:,} [{neg/len(d)*100:.1f}%])")

---
## Step 5 — Preprocessing (fit on `df_train` only, transform all)

⚠️ Every transformer below is fitted **exclusively** on `df_train`, then
applied to `df_val` and `df_test`. This prevents data leakage.

In [ ]:
# --- Define feature groups ---

# Categorical features
cat_cols = [c for c in ['race', 'insurance', 'admission_type', 'admission_location', 'first_careunit']
            if c in df_features.columns]

# Binary / indicator columns (missing flags, Charlson components, ICD flags)
binary_cols = (
    [c for c in df_features.columns if c.endswith('_missing')]
    + list(CHARLSON_MAP.keys())
    + [c for c in df_features.columns if c.startswith('icd_')]
)
binary_cols = [c for c in binary_cols if c in df_features.columns]

# ID / target / meta columns to exclude from features
exclude_cols = {'subject_id', 'hadm_id', 'los_target'}

# Continuous features = everything not categorical, binary, or excluded
continuous_cols = [
    c for c in df_features.columns
    if c not in set(cat_cols) | set(binary_cols) | exclude_cols
    and df_features[c].dtype in ['float64', 'float32', 'int64', 'int32', 'Int64']
]

print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Continuous features ({len(continuous_cols)}): {continuous_cols[:10]}...")
print(f"Binary features ({len(binary_cols)}): {binary_cols[:10]}...")

In [ ]:
# --- Categorical: fill missing → OneHotEncode ---
for c in cat_cols:
    df_train[c] = df_train[c].fillna('Unknown').astype(str)
    df_val[c]   = df_val[c].fillna('Unknown').astype(str)
    df_test[c]  = df_test[c].fillna('Unknown').astype(str)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[cat_cols])

ohe_train = pd.DataFrame(ohe.transform(df_train[cat_cols]),
                         columns=ohe.get_feature_names_out(cat_cols),
                         index=df_train.index)
ohe_val   = pd.DataFrame(ohe.transform(df_val[cat_cols]),
                         columns=ohe.get_feature_names_out(cat_cols),
                         index=df_val.index)
ohe_test  = pd.DataFrame(ohe.transform(df_test[cat_cols]),
                         columns=ohe.get_feature_names_out(cat_cols),
                         index=df_test.index)

print(f"OHE features: {ohe_train.shape[1]} columns")

In [ ]:
# --- Continuous: impute (median) → scale ---
cont_imputer = SimpleImputer(strategy='median')
cont_imputer.fit(df_train[continuous_cols])

cont_train = pd.DataFrame(cont_imputer.transform(df_train[continuous_cols]),
                          columns=continuous_cols, index=df_train.index)
cont_val   = pd.DataFrame(cont_imputer.transform(df_val[continuous_cols]),
                          columns=continuous_cols, index=df_val.index)
cont_test  = pd.DataFrame(cont_imputer.transform(df_test[continuous_cols]),
                          columns=continuous_cols, index=df_test.index)

scaler = StandardScaler()
scaler.fit(cont_train)

cont_train = pd.DataFrame(scaler.transform(cont_train),
                          columns=continuous_cols, index=df_train.index)
cont_val   = pd.DataFrame(scaler.transform(cont_val),
                          columns=continuous_cols, index=df_val.index)
cont_test  = pd.DataFrame(scaler.transform(cont_test),
                          columns=continuous_cols, index=df_test.index)

print(f"Continuous features: {cont_train.shape[1]} columns (imputed + scaled)")

In [ ]:
# --- Binary: impute (most frequent) ---
bin_imputer = SimpleImputer(strategy='most_frequent')
bin_imputer.fit(df_train[binary_cols])

bin_train = pd.DataFrame(bin_imputer.transform(df_train[binary_cols]),
                         columns=binary_cols, index=df_train.index)
bin_val   = pd.DataFrame(bin_imputer.transform(df_val[binary_cols]),
                         columns=binary_cols, index=df_val.index)
bin_test  = pd.DataFrame(bin_imputer.transform(df_test[binary_cols]),
                         columns=binary_cols, index=df_test.index)

print(f"Binary features: {bin_train.shape[1]} columns (imputed)")

In [ ]:
# --- Assemble final feature matrices ---

X_train = pd.concat([cont_train, bin_train, ohe_train], axis=1)
X_val   = pd.concat([cont_val,   bin_val,   ohe_val],   axis=1)
X_test  = pd.concat([cont_test,  bin_test,  ohe_test],  axis=1)

y_train = df_train['los_target'].values
y_val   = df_val['los_target'].values
y_test  = df_test['los_target'].values

# Keep feature names for later
feature_names = list(X_train.columns)

print(f"Final feature matrices:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

# Save fitted transformers
preprocessor = {
    'ohe': ohe,
    'cont_imputer': cont_imputer,
    'scaler': scaler,
    'bin_imputer': bin_imputer,
    'cat_cols': cat_cols,
    'continuous_cols': continuous_cols,
    'binary_cols': binary_cols,
    'feature_names': feature_names,
}
with open(PROCESSED / 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print(f"\nSaved preprocessor to {PROCESSED / 'preprocessor.pkl'}")

---
## Step 6 — Data Analysis & Plots (on training data only)

⚠️ All exploratory analysis uses only the training set. We never peek at
test data to avoid biasing our modeling decisions.

In [ ]:
# Use df_train (pre-scaling) for interpretable plots
df_plot = df_train.copy()

# --- 1. Missing value heatmap (top 20 by missingness) ---
missing_pct = (df_plot.isnull().sum() / len(df_plot) * 100).sort_values(ascending=False)
top_missing = missing_pct[missing_pct > 0].head(20)

if len(top_missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    top_missing.plot.barh(ax=ax, color='salmon')
    ax.set_xlabel('Missing %')
    ax.set_title('Top 20 Features by Missing Value %')
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig(FIGURES / 'missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No missing values in training set.")

In [ ]:
# --- 2. Class distribution ---
fig, ax = plt.subplots(figsize=(6, 4))
df_plot['los_target'].value_counts().sort_index().plot.bar(
    ax=ax, color=['steelblue', 'coral'],
    edgecolor='black'
)
ax.set_xticklabels(['LOS ≤ 7d', 'LOS > 7d'], rotation=0)
ax.set_ylabel('Count')
ax.set_title('Class Distribution (Training Set)')
for i, v in enumerate(df_plot['los_target'].value_counts().sort_index()):
    ax.text(i, v + len(df_plot)*0.005, f"{v:,}", ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. Age distribution by class ---
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, 'steelblue'), (1, 'coral')]:
    subset = df_plot[df_plot['los_target'] == label]['anchor_age']
    ax.hist(subset, bins=40, alpha=0.6, color=color,
            label='LOS ≤ 7d' if label == 0 else 'LOS > 7d', edgecolor='black')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.set_title('Age Distribution by LOS Class')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 4. LOS distribution (clipped at 30 days) ---
if 'los_days' in df_plot.columns:
    los_col = 'los_days'
else:
    # Reconstruct from admit_times if needed
    los_col = None
    print("LOS days column dropped (expected — it was used to create target).")

# We can reconstruct from the original cohort
los_train_vals = df.set_index('hadm_id').loc[
    df_train['hadm_id'], 'los_days'
].values if 'los_days' in df.columns else None

# The los_days column was dropped earlier; let's use the stored values
# Actually we dropped it in drop_cols — let's recalculate from admit_times
# For the plot, we need the original cohort df which still has los_days
# Since df still exists in memory:
if los_train_vals is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    clipped = np.clip(los_train_vals, 0, 30)
    ax.hist(clipped, bins=60, color='teal', edgecolor='black', alpha=0.7)
    ax.axvline(x=7, color='red', linestyle='--', linewidth=2, label='7-day threshold')
    ax.set_xlabel('Length of Stay (days, clipped at 30)')
    ax.set_ylabel('Count')
    ax.set_title('LOS Distribution (Training Set)')
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES / 'los_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 5. Correlation heatmap (top 20 numeric features) ---
numeric_train = df_plot.select_dtypes(include=[np.number])
# Pick top 20 by variance (excluding binary-only)
variances = numeric_train.var().sort_values(ascending=False)
top20_vars = variances.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
corr = numeric_train[top20_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Top 20 Numeric Features (Training Set)')
plt.tight_layout()
fig.savefig(FIGURES / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 6. Vital signs boxplots by LOS class ---
box_vitals = ['heart_rate_mean', 'spo2_mean', 'creatinine_mean']
available_box = [c for c in box_vitals if c in df_plot.columns]

if available_box:
    fig, axes = plt.subplots(1, len(available_box), figsize=(5*len(available_box), 5))
    if len(available_box) == 1:
        axes = [axes]
    for ax, col in zip(axes, available_box):
        data_0 = df_plot.loc[df_plot['los_target'] == 0, col].dropna()
        data_1 = df_plot.loc[df_plot['los_target'] == 1, col].dropna()
        ax.boxplot([data_0, data_1], labels=['LOS ≤ 7d', 'LOS > 7d'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightblue'))
        ax.set_title(col)
        ax.set_ylabel('Value')
    plt.suptitle('Vital Signs / Labs by LOS Class (Training Set)', y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES / 'vitals_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 7. Charlson score distribution by class ---
if 'charlson_score' in df_plot.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, color in [(0, 'steelblue'), (1, 'coral')]:
        subset = df_plot[df_plot['los_target'] == label]['charlson_score']
        ax.hist(subset, bins=range(0, 20), alpha=0.6, color=color,
                label='LOS ≤ 7d' if label == 0 else 'LOS > 7d', edgecolor='black')
    ax.set_xlabel('Charlson Comorbidity Score')
    ax.set_ylabel('Count')
    ax.set_title('Charlson Score Distribution by LOS Class (Training Set)')
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES / 'charlson_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 8. Top 15 ICD chapter frequencies ---
icd_flag_cols = [c for c in df_plot.columns if c.startswith('icd_')]
if icd_flag_cols:
    icd_freqs = df_plot[icd_flag_cols].sum().sort_values(ascending=True).tail(15)
    fig, ax = plt.subplots(figsize=(8, 6))
    icd_freqs.plot.barh(ax=ax, color='mediumseagreen', edgecolor='black')
    ax.set_xlabel('Patient Count')
    ax.set_title('Top 15 ICD Chapter Frequencies (Training Set)')
    plt.tight_layout()
    fig.savefig(FIGURES / 'icd_chapter_frequencies.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Step 7 — MLP Model (PyTorch)

A 3-layer feedforward network with BatchNorm and Dropout.
We use `BCEWithLogitsLoss` with class-weight correction for the imbalanced target.

In [ ]:
# --- Model definition ---

class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# --- Prepare data ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val.values, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)

# --- Loss with class weighting ---
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f"Class weights — pos_weight: {pos_weight.item():.2f} (neg={n_neg:,}, pos={n_pos:,})")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# --- Model, optimizer, scheduler ---
model = MLPClassifier(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

print(f"\nModel architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# --- Training loop ---
N_EPOCHS = 50
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(1, N_EPOCHS + 1):
    # Train
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_loss = epoch_loss / n_batches
    train_losses.append(train_loss)

    # Validate
    model.eval()
    val_loss = 0.0
    n_val_batches = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
            n_val_batches += 1
    val_loss /= n_val_batches
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), PROCESSED / 'best_model.pt')
        marker = ' ✓ saved'
    else:
        marker = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{N_EPOCHS}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{marker}")

print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# --- Loss curve ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, N_EPOCHS+1), train_losses, label='Train Loss', linewidth=2)
ax.plot(range(1, N_EPOCHS+1), val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('BCEWithLogitsLoss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES / 'loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Evaluation (on `df_test` only)

We load the best checkpoint and evaluate on the held-out test set.

In [ ]:
# Load best model
model.load_state_dict(torch.load(PROCESSED / 'best_model.pt', map_location=device, weights_only=True))
model.eval()

# Get predictions on test set
X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)
with torch.no_grad():
    logits_test = model(X_test_t).cpu().numpy()

probs_test = 1 / (1 + np.exp(-logits_test))  # sigmoid
preds_test = (probs_test >= 0.5).astype(int)

# --- Metrics ---
auroc    = roc_auc_score(y_test, probs_test)
auprc    = average_precision_score(y_test, probs_test)
f1       = f1_score(y_test, preds_test)
acc      = accuracy_score(y_test, preds_test)
bal_acc  = balanced_accuracy_score(y_test, preds_test)

print("=" * 50)
print("  TEST SET EVALUATION")
print("=" * 50)
print(f"  AUROC:             {auroc:.4f}")
print(f"  AUPRC:             {auprc:.4f}")
print(f"  F1 Score:          {f1:.4f}")
print(f"  Accuracy:          {acc:.4f}")
print(f"  Balanced Accuracy: {bal_acc:.4f}")

In [ ]:
# --- Confusion matrix (normalized) ---
cm = confusion_matrix(y_test, preds_test, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
            xticklabels=['LOS ≤ 7d', 'LOS > 7d'],
            yticklabels=['LOS ≤ 7d', 'LOS > 7d'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Normalized Confusion Matrix (Test Set)')
plt.tight_layout()
fig.savefig(FIGURES / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- ROC + Precision-Recall curves ---
fpr, tpr, _ = roc_curve(y_test, probs_test)
precision, recall, _ = precision_recall_curve(y_test, probs_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax1.plot(fpr, tpr, linewidth=2, label=f'AUROC = {auroc:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# PR
ax2.plot(recall, precision, linewidth=2, label=f'AUPRC = {auprc:.3f}')
baseline = y_test.mean()
ax2.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Baseline = {baseline:.3f}')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Calibration curve ---

fraction_pos, mean_predicted = calibration_curve(y_test, probs_test, n_bins=10, strategy='uniform')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_predicted, fraction_pos, 's-', linewidth=2, label='MLP')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES / 'calibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Permutation feature importance (top 15) ---
print("Computing permutation importance (this may take a few minutes)...")

def compute_permutation_importance(model, X, y, n_repeats=10, random_state=42):
    """Manual permutation importance using AUROC."""
    rng = np.random.RandomState(random_state)
    model.eval()
    
    # Baseline score
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()
    baseline = roc_auc_score(y, 1 / (1 + np.exp(-logits)))
    
    importances = np.zeros((X.shape[1], n_repeats))
    for col in range(X.shape[1]):
        for r in range(n_repeats):
            X_perm = X.copy()
            X_perm[:, col] = rng.permutation(X_perm[:, col])
            with torch.no_grad():
                logits = model(torch.tensor(X_perm, dtype=torch.float32).to(device)).cpu().numpy()
            score = roc_auc_score(y, 1 / (1 + np.exp(-logits)))
            importances[col, r] = baseline - score
        if (col + 1) % 20 == 0:
            print(f"  Processed {col + 1}/{X.shape[1]} features...")
    
    return importances.mean(axis=1), importances.std(axis=1)

imp_mean, imp_std = compute_permutation_importance(model, X_test.values, y_test)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': imp_mean,
    'importance_std': imp_std,
}).sort_values('importance_mean', ascending=False)

top15 = importance_df.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15['feature'], top15['importance_mean'],
        xerr=top15['importance_std'], color='darkorange', edgecolor='black')
ax.set_xlabel('Mean AUROC Decrease')
ax.set_title('Permutation Feature Importance \u2014 Top 15 (Test Set)')
plt.tight_layout()
fig.savefig(FIGURES / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### Fairness Check

We compute AUROC separately for demographic subgroups to check for disparities.

In [ ]:
# --- Fairness: AUROC by subgroup ---

fairness_results = []

# Gender
if 'gender' in df_test.columns:
    for g_val, g_label in [(1, 'Male'), (0, 'Female')]:
        mask = df_test['gender'].values == g_val
        if mask.sum() > 10 and y_test[mask].nunique() if isinstance(y_test, pd.Series) else len(set(y_test[mask])) > 1:
            auc = roc_auc_score(y_test[mask], probs_test[mask])
            fairness_results.append({'Group': f'Gender: {g_label}', 'N': mask.sum(), 'AUROC': f'{auc:.4f}'})

# Race
if 'race' in df_test.columns:
    for race in df_test['race'].unique():
        mask = df_test['race'].values == race
        if mask.sum() >= 50:
            y_sub = y_test[mask]
            if len(set(y_sub)) > 1:
                auc = roc_auc_score(y_sub, probs_test[mask])
                fairness_results.append({'Group': f'Race: {race}', 'N': mask.sum(), 'AUROC': f'{auc:.4f}'})

# Insurance
if 'insurance' in df_test.columns:
    for ins in df_test['insurance'].unique():
        mask = df_test['insurance'].values == ins
        if mask.sum() >= 50:
            y_sub = y_test[mask]
            if len(set(y_sub)) > 1:
                auc = roc_auc_score(y_sub, probs_test[mask])
                fairness_results.append({'Group': f'Insurance: {ins}', 'N': mask.sum(), 'AUROC': f'{auc:.4f}'})

if fairness_results:
    fairness_df = pd.DataFrame(fairness_results)
    print("\nFairness Check — AUROC by Subgroup:")
    display(fairness_df)
else:
    print("Could not compute subgroup fairness (demographic columns may have been encoded).")
    print("Note: race/insurance were one-hot encoded. Reconstruct from df_test for subgroup analysis.")

---
## Summary

### Key Findings
- **Task:** Binary classification of Length of Stay > 7 days using MIMIC-IV data.
- **Cohort:** Adult patients (≥18), first admission only, LOS ≥ 4 hours.
- **Features:** Demographics, first-24h vitals (8 vital signs × 4 aggregations),
  first-24h labs (8 lab tests × 3 aggregations), Charlson Comorbidity Index
  (17 components + weighted score), and top ICD chapter flags.
- **Model:** 3-layer MLP with BatchNorm and Dropout, trained with class-weighted
  BCEWithLogitsLoss.

### Top Predictors
See the permutation importance plot above. Typically, Charlson score, age,
creatinine, and vital sign variability (std) are strong predictors of extended LOS.

### Limitations
- **First 24h only:** Features are limited to the first day of admission.
  Clinicians make ongoing assessments that our model cannot capture.
- **Static model:** An MLP treats all features as a flat vector. It cannot
  capture temporal dynamics within the 24h window.
- **Missing data:** Some patients lack ICU charted vitals (non-ICU admissions).
  The missing-indicator pattern partially addresses this but is imperfect.
- **Single-site:** MIMIC-IV is from a single academic medical center (BIDMC).
  External validation on multi-site data is essential before deployment.
- **No textual data:** Discharge notes, nursing notes, and radiology reports
  contain rich prognostic information that we do not use here.

### Next Steps
- **Temporal models:** LSTM or Transformer on hourly vital/lab time series.
- **Clinical text:** Fine-tune ClinicalBERT on discharge/nursing notes.
- **Multi-task learning:** Predict LOS as both a regression and classification target.
- **External validation:** Test on eICU or MIMIC-III for generalizability.
- **Calibration:** Apply Platt scaling or isotonic regression for better-calibrated
  probability estimates.

In [ ]:
# --- Pipeline runtime ---
elapsed = time.time() - PIPELINE_START
hours, remainder = divmod(elapsed, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"\nTotal pipeline runtime: {int(hours)}h {int(minutes)}m {seconds:.1f}s")